In [1]:
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np
from textblob import TextBlob
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# ======================================
# Load Model & Vectorizer (once)
# ======================================
model = joblib.load("logistic_model.pkl")
vectorizer = joblib.load("tfidf_vectorizer.pkl")

app = FastAPI(title="Fake News Detection API")

# ======================================
# Request Body Schema
# ======================================
class NewsRequest(BaseModel):
    text: str

# ======================================
# Feature Engineering
# ======================================
def compute_features(text):
    words = text.split()
    word_count = len(words)

    unique_word_ratio = len(set(words)) / word_count if word_count > 0 else 0

    avg_word_len = (
        np.mean([len(w) for w in words]) if word_count > 0 else 0
    )

    stopword_ratio = (
        sum(1 for w in words if w.lower() in ENGLISH_STOP_WORDS) / word_count
        if word_count > 0 else 0
    )

    blob = TextBlob(text)
    polarity = blob.sentiment.polarity
    subjectivity = blob.sentiment.subjectivity

    return {
        "word_count": word_count,
        "unique_word_ratio": unique_word_ratio,
        "avg_word_len": avg_word_len,
        "stopword_ratio": stopword_ratio,
        "polarity": polarity,
        "subjectivity": subjectivity,
    }

# ======================================
# Prediction Endpoint
# ======================================
@app.post("/predict")
def predict(news: NewsRequest):
    text = news.text

    X = vectorizer.transform([text])
    prediction = model.predict(X)[0]
    probabilities = model.predict_proba(X)[0]

    features = compute_features(text)

    return {
        "label": "FAKE" if prediction == 1 else "REAL",
        "confidence": float(max(probabilities)),
        "fake_probability": float(probabilities[1]),
        "real_probability": float(probabilities[0]),
        **features
    }


C:\Users\91903\anaconda3\envs\nlp_course_env\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\91903\anaconda3\envs\nlp_course_env\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.7.2 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\91903\anaconda3\envs\nlp_course_env\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from 